In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
warnings.filterwarnings('ignore')

class StockPricePredictor:
    def __init__(self,df):
        self.model = None
        self.scaler = StandardScaler()
        self.feature_cols = [col for col in df.columns if col not in ['date']]

        
    def create_target(self, df, price_col='c'):
        """
        Create next day's closing price as target
        """
        # df['target'] = df[price_col].shift(-1)
        df['target'] = (df['c'].shift(-5) > df['c']).astype(int)
        extended_features = self.feature_cols 
        
        # Keep only available features
        available_features = [col for col in extended_features if col in df.columns]
        return df.dropna()
    

    
    def prepare_features(self, df):
        """
        Prepare features for training
        """
        # Add technical indicators
        # df = self.add_technical_indicators(df)
        
        # Create target variable
        df = self.create_target(df)
        
        # Define extended feature set
        extended_features = self.feature_cols 
        
        # # Keep only available features
        available_features = [col for col in extended_features if col in df.columns]
        
        return df, available_features
    
    def train_model(self, df, test_size=0.2):
        """
        Train the gradient boosted trees model
        """
        # Prepare features
        df_processed, features = self.prepare_features(df)
        if len(df_processed) == 0:
            raise ValueError("No data available after preprocessing. Check for NaN values in your features.")
        
        print(f"Using features: {features}")
        print(f"Data shape: {df_processed.shape}")
        
        # Split data (time series split)
        split_idx = int(len(df_processed) * (1 - test_size))
        train_df = df_processed.iloc[:split_idx]
        test_df = df_processed.iloc[split_idx:]
        
        # # Prepare features and target
        X_train = train_df[features]
        print(X_train)
        y_train = train_df['target']
        print(y_train)
        X_test = test_df[features]
        y_test = test_df['target']
        # X = df_processed[features]
        # y = df_processed['target']

        # X_train, X_test, y_train, y_test = train_test_split(
        # X, y, 
        # test_size=test_size, 
        # random_state=42,
        # shuffle=True)
    
        
        # Scale features
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Initialize and train model
        # self.model = GradientBoostingClassifier(
        # n_estimators=100,
        # learning_rate=0.1,
        # max_depth=3,
        # min_samples_split=20,
        # min_samples_leaf=10,
        # subsample=0.8,
        # random_state=42)
        self.model = GradientBoostingClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        min_samples_leaf=20,
        random_state=42)


        
        self.model.fit(X_train_scaled, y_train)
        train_pred_proba = self.model.predict_proba(X_train_scaled)[:, 1]
        test_pred_proba = self.model.predict_proba(X_test_scaled)[:, 1]
        
        train_pred_classes = (train_pred_proba >= 0.5).astype(int)
        test_pred_classes = (test_pred_proba >= 0.5).astype(int)
        train_acc = accuracy_score(y_train, train_pred_classes)
        test_acc = accuracy_score(y_test, test_pred_classes)
        train_prec = precision_score(y_train, train_pred_classes, zero_division=0)
        test_prec = precision_score(y_test, test_pred_classes, zero_division=0)
        train_rec = recall_score(y_train, train_pred_classes, zero_division=0)
        test_rec = recall_score(y_test, test_pred_classes, zero_division=0)
        train_f1 = f1_score(y_train, train_pred_classes)
        test_f1 = f1_score(y_test, test_pred_classes)
        train_auc = roc_auc_score(y_train, train_pred_proba)
        test_auc = roc_auc_score(y_test, test_pred_proba)

# Display results
        print("\nModel Performance (Binary Classification):")
        print(f"Train Accuracy:  {train_acc:.4f}")
        print(f"Test Accuracy:   {test_acc:.4f}")
        print(f"Train Precision: {train_prec:.4f}")
        print(f"Test Precision:  {test_prec:.4f}")
        print(f"Train Recall:    {train_rec:.4f}")
        print(f"Test Recall:     {test_rec:.4f}")
        print(f"Train F1-score:  {train_f1:.4f}")
        print(f"Test F1-score:   {test_f1:.4f}")
        print(f"Train AUC:       {train_auc:.4f}")
        print(f"Test AUC:        {test_auc:.4f}")
        
        print("\nConfusion Matrix (Test):")
        print(confusion_matrix(y_test, test_pred_classes))
        
        # Return all useful objects
        return {
            'X_train': X_train_scaled, 'y_train': y_train,
            'X_test': X_test_scaled, 'y_test': y_test,
            'train_pred': train_pred_classes, 'test_pred': test_pred_classes,
            'train_pred_proba': train_pred_proba, 'test_pred_proba': test_pred_proba,
            'features': features
        }


    
    
    # def predict_next_day(self, df):
    #     """
    #     Predict next day's price using the most recent data
    #     """
    #     if self.model is None:
    #         raise ValueError("Model not trained yet. Call train_model first.")
        
    #     # Prepare features for the most recent row
    #     df_processed, features = self.prepare_features(df)
    #     recent_data = df_processed[features].iloc[-1:].copy()
        
    #     # Scale features
    #     recent_data_scaled = self.scaler.transform(recent_data)
        
    #     # Make prediction
    #     prediction = self.model.predict(recent_data_scaled)[0]
        
    #     return prediction
    
    # def feature_importance(self, features):
    #     """
    #     Display feature importance
    #     """
    #     if self.model is None:
    #         raise ValueError("Model not trained yet.")
        
    #     importance_df = pd.DataFrame({
    #         'feature': features,
    #         'importance': self.model.feature_importances_
    #     }).sort_values('importance', ascending=False)
        
    #     plt.figure(figsize=(10, 6))
    #     plt.barh(importance_df['feature'], importance_df['importance'])
    #     plt.xlabel('Feature Importance')
    #     plt.title('Gradient Boosting Feature Importance')
    #     plt.gca().invert_yaxis()
    #     plt.tight_layout()
    #     plt.show()
        
    #     return importance_df
    
    # def plot_predictions(self, results):
    #     """
    #     Plot actual vs predicted prices
    #     """
    #     plt.figure(figsize=(15, 5))
        
    #     # Plot test predictions
    #     plt.subplot(1, 2, 1)
    #     plt.plot(results['y_test'].values, label='Actual', alpha=0.7)
    #     plt.plot(results['test_pred'], label='Predicted', alpha=0.7)
    #     plt.title('Test Set: Actual vs Predicted Prices')
    #     plt.xlabel('Time')
    #     plt.ylabel('Price')
    #     plt.legend()
        
    #     # Plot scatter plot
    #     plt.subplot(1, 2, 2)
    #     plt.scatter(results['y_test'].values, results['test_pred'], alpha=0.5)
    #     plt.plot([results['y_test'].min(), results['y_test'].max()], 
    #             [results['y_test'].min(), results['y_test'].max()], 'r--')
    #     plt.xlabel('Actual Prices')
    #     plt.ylabel('Predicted Prices')
    #     plt.title('Predicted vs Actual Prices')
        
    #     plt.tight_layout()
    #     plt.show()

In [16]:


# Initialize and train the model
# predictor = StockPricePredictor()

# # Create sample data (replace with your actual data)
# df = pd.read_csv('NEPSE_ADX_data.csv')
# df['date'] = pd.to_datetime(df['date'])
# df = df.sort_values('date').reset_index(drop=True)
# df = df[df['date'] > '2013-01-01'].reset_index(drop=True)
# df.drop('s', axis=1, inplace=True)

# # Train the model
# results = predictor.train_model(df, test_size=0.2)

# Show feature importance
# importance_df = predictor.feature_importance(results['features'])
# print("\nTop 10 Most Important Features:")
# print(importance_df.head(10))

# # Plot predictions
# predictor.plot_predictions(results)

# # Predict next day's price
# next_day_prediction = predictor.predict_next_day(df)
# print(f"\nNext day predicted price: {next_day_prediction:.2f}")

In [17]:
# importance_df = predictor.feature_importance(results['features'])
# print("\nTop 10 Most Important Features:")
# print(importance_df.head(10))

NOW TIME TO CREATE SOME LAGS VARIABLE

In [19]:
import pandas as pd

# Suppose your dataframe is named `df` and already sorted by date ascending
# Columns: ['open', 'high', 'low', 'close', 'volume', 'psar', 'posDI', 'negDI', 'adx', 'accumulated_sentiment']

# Ensure the dataframe is sorted by date
df = pd.read_csv('NEPSE_ADX_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df = df[df['date'] > '2013-01-01'].reset_index(drop=True)
df.drop('s', axis=1, inplace=True)
df = df.sort_values('date').reset_index(drop=True)

# List of features to create lags for
features = ['o', 'h', 'l', 'c', 'v', 
            'psar', '+DI', '-DI', 'ADX']

# Create lag features up to 4 days back
max_lag = 4
for lag in range(1, max_lag + 1):
    lagged = df[features].shift(lag)
    lagged.columns = [f"{col}_lag{lag}" for col in features]
    df = pd.concat([df, lagged], axis=1)

# Drop rows with NaN (the first 4 rows won’t have full lag data)
df = df.dropna().reset_index(drop=True)

print(df.head())



       o      h      l      c    v       date        psar    +DI    -DI  \
0  529.0  529.0  529.0  529.0  0.0 2013-01-08  540.760000    0.0    0.0   
1  525.0  525.0  525.0  525.0  0.0 2013-01-09  540.524800    0.0  100.0   
2  529.0  529.0  529.0  529.0  0.0 2013-01-10  539.903808  100.0    0.0   
3  530.0  530.0  530.0  530.0  0.0 2013-01-13  539.307656  100.0    0.0   
4  530.0  530.0  530.0  530.0  0.0 2013-01-15  538.735349    0.0    0.0   

     ADX  ...  ADX_lag3  o_lag4  h_lag4  l_lag4  c_lag4  v_lag4   psar_lag4  \
0    0.0  ...     100.0   532.0   532.0   532.0   532.0     0.0  521.968604   
1  100.0  ...     100.0   534.0   534.0   534.0   534.0     0.0  525.394255   
2  100.0  ...     100.0   532.0   532.0   532.0   532.0     0.0  528.203289   
3  100.0  ...       0.0   529.0   529.0   529.0   529.0     0.0  541.000000   
4    0.0  ...     100.0   529.0   529.0   529.0   529.0     0.0  540.760000   

   +DI_lag4  -DI_lag4  ADX_lag4  
0     100.0       0.0     100.0  
1     

In [20]:
predictor = StockPricePredictor(df)
# Train the model
results = predictor.train_model(df, test_size=0.2)

Using features: ['o', 'h', 'l', 'c', 'v', 'psar', '+DI', '-DI', 'ADX', 'o_lag1', 'h_lag1', 'l_lag1', 'c_lag1', 'v_lag1', 'psar_lag1', '+DI_lag1', '-DI_lag1', 'ADX_lag1', 'o_lag2', 'h_lag2', 'l_lag2', 'c_lag2', 'v_lag2', 'psar_lag2', '+DI_lag2', '-DI_lag2', 'ADX_lag2', 'o_lag3', 'h_lag3', 'l_lag3', 'c_lag3', 'v_lag3', 'psar_lag3', '+DI_lag3', '-DI_lag3', 'ADX_lag3', 'o_lag4', 'h_lag4', 'l_lag4', 'c_lag4', 'v_lag4', 'psar_lag4', '+DI_lag4', '-DI_lag4', 'ADX_lag4']
Data shape: (2918, 47)
FUkc this world


            o        h        l        c             v         psar  \
0      529.00   529.00   529.00   529.00  0.000000e+00   540.760000   
1      525.00   525.00   525.00   525.00  0.000000e+00   540.524800   
2      529.00   529.00   529.00   529.00  0.000000e+00   539.903808   
3      530.00   530.00   530.00   530.00  0.000000e+00   539.307656   
4      530.00   530.00   530.00   530.00  0.000000e+00   538.735349   
...       ...      ...      ...      ...           ...          ..